# Inverse-RL Colab Notebook

Single resumable notebook for the inverse-RL experiment. Cells 0–3 are implemented; cells 4–8 are intentionally left as labeled stubs for later execution-guide steps.

In [ ]:
# Cell 0 — setup (repo clone, dependencies, GPU)
import os
import sys
import subprocess
from pathlib import Path

GIT_URL = os.environ.get("INVERSE_RL_GIT_URL", "https://github.com/REPLACE_ME/inverse-RL.git")
REPO_DIR = Path("/content/inverse-RL")

if REPO_DIR.exists():
    print(f"[skip] repo already exists at {REPO_DIR}")
else:
    print(f"[run] cloning {GIT_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", GIT_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("[run] installing requirements")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA GPU FOUND"
print(f"GPU: {gpu_name}")


In [ ]:
# Cell 1 — Drive + config + resumable state
import json
import os
from pathlib import Path

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/inverse-rl")
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "ckpts"
RESULTS_DIR = ROOT / "results"
for d in (ROOT, DATA_DIR, CKPT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    "shakedown_model": "meta-llama/Llama-3.2-1B-Instruct",
    "lora_r": 32,
    "G": 8,
    "max_prompt_len": 2048,
    "max_completion_len": 512,
    "lr": 2e-6,
    "train_levels": [1],
    "composition_train_levels": [2],
    "eval_levels": [1, 2, 3, 4],
    "pass_at_k": 8,
    "seeds": [0, 1, 2],
    # Dataset sizes are intentionally configurable for shakedowns vs. full runs.
    "n_forward_l1": 2500,
    "n_eval_per_cell": 100,
    "n_comp_train": 2500,
    "n_inverse_train": 2500,
    "n_inverse_l1to2_seen": 1200,
}

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("[skip] HF_TOKEN not found in Colab userdata; gated model downloads may fail")

STATE_PATH = ROOT / "state.json"

def state_load():
    if STATE_PATH.exists():
        with STATE_PATH.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {}

state = state_load()

def state_save(key=None, value=True):
    global state
    state = state_load()
    if key is not None:
        state[key] = value
    tmp = STATE_PATH.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, sort_keys=True)
    tmp.replace(STATE_PATH)
    return state

def exists(path):
    return Path(path).exists()

def done(key):
    return bool(state_load().get(key, False))

state_save()
print(f"ROOT={ROOT}")
print(f"state={STATE_PATH}")


In [ ]:
# Cell 2 — data generation (resumable JSONL outputs)
import subprocess
import sys
from pathlib import Path

DATASETS = {
    "fwd_l1_all.jsonl": ["--task", "forward", "--levels", "1", "--n", str(CFG["n_forward_l1"]), "--pool", "all", "--seed", str(CFG["seeds"][0])],
    "fwd_l1to4_eval.jsonl": ["--task", "forward", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][0])],
    "fwd_l2_seen_comp_train.jsonl": ["--task", "forward", "--levels", "2", "--n", str(CFG["n_comp_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1_seen_train.jsonl": ["--task", "inverse", "--levels", "1", "--n", str(CFG["n_inverse_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1to4_eval.jsonl": ["--task", "inverse", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][2])],
    "inv_l1to2_seen_optional.jsonl": ["--task", "inverse", "--levels", "1,2", "--n", str(CFG["n_inverse_l1to2_seen"]), "--pool", "seen", "--seed", str(CFG["seeds"][2])],
}

missing = [name for name in DATASETS if not (DATA_DIR / name).exists()]
if done("data") and not missing:
    print(f"[skip] data already present in {DATA_DIR}")
else:
    print(f"[run] generating {len(missing)} missing dataset(s) in {DATA_DIR}")
    for name in missing:
        out = DATA_DIR / name
        cmd = [sys.executable, "inverse_tasks.py", *DATASETS[name], "--out", str(out)]
        print("[run]", " ".join(cmd))
        subprocess.run(cmd, check=True)
    state_save("data", True)
    print("[run] data generation complete")


In [ ]:
# Cell 3 — evaluation utilities (HF/LoRA loading, vLLM generation, CSV outputs)
import gc
import json
import re
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

from inverse_tasks import ALL_SKILLS
from skills_inverse import SKILLS
from verifier import forward_reward, inverse_reward

_ACTIVE_MODEL = None
_ACTIVE_LLM = None
_ACTIVE_LLM_KEY = None

def _safe_name(path_or_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(path_or_name).rstrip("/"))[-120:]

def _read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _model_key(model):
    if isinstance(model, dict):
        return model["model_path"]
    return str(model)

def load_model(path_or_name, adapter=None):
    """Return a vLLM-ready model descriptor; merge an optional LoRA adapter once."""
    if adapter is None:
        model = {"model_path": str(path_or_name), "adapter": None, "name": _safe_name(path_or_name)}
        print(f"[skip] using base/merged model {model['model_path']}")
        return model

    merged_dir = CKPT_DIR / f"merged_{_safe_name(path_or_name)}__{_safe_name(adapter)}"
    if merged_dir.exists():
        print(f"[skip] merged adapter already exists at {merged_dir}")
        return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

    print(f"[run] merging LoRA adapter {adapter} into {path_or_name}")
    tokenizer = AutoTokenizer.from_pretrained(path_or_name, token=HF_TOKEN)
    base = AutoModelForCausalLM.from_pretrained(
        path_or_name,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    merged = PeftModel.from_pretrained(base, adapter).merge_and_unload()
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    del merged, base, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

def _get_llm(model):
    global _ACTIVE_LLM, _ACTIVE_LLM_KEY
    key = _model_key(model)
    if _ACTIVE_LLM is not None and _ACTIVE_LLM_KEY == key:
        return _ACTIVE_LLM
    if _ACTIVE_LLM is not None:
        del _ACTIVE_LLM
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"[run] loading vLLM model {key}")
    _ACTIVE_LLM = LLM(
        model=key,
        tokenizer=key,
        trust_remote_code=True,
        dtype="bfloat16" if torch.cuda.is_available() else "float32",
        max_model_len=CFG["max_prompt_len"] + CFG["max_completion_len"],
        gpu_memory_utilization=0.90,
    )
    _ACTIVE_LLM_KEY = key
    return _ACTIVE_LLM

def generate(prompts, n=1, model=None):
    """Generate n completions per prompt with vLLM; call eval_task/load_model first or pass model."""
    global _ACTIVE_MODEL
    if model is not None:
        _ACTIVE_MODEL = model
    if _ACTIVE_MODEL is None:
        raise ValueError("No active model. Pass model=... or call eval_task(model, ...) first.")
    llm = _get_llm(_ACTIVE_MODEL)
    params = SamplingParams(
        n=n,
        temperature=0.7 if n > 1 else 0.0,
        top_p=0.95,
        max_tokens=CFG["max_completion_len"],
    )
    outputs = llm.generate(prompts, params)
    return [[choice.text for choice in out.outputs] for out in outputs]

def _cells_from_arg(cells):
    if isinstance(cells, (str, Path)):
        return _read_jsonl(cells)
    return list(cells)

def eval_task(model, cells, task, k=None, out_name=None):
    """Evaluate a task and save tidy metrics: level, split, pass@1, pass@k."""
    global _ACTIVE_MODEL
    k = int(k or CFG["pass_at_k"])
    problems = _cells_from_arg(cells)
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"eval_{task}_{model_name}_k{k}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing eval results from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] evaluating {task} on {len(problems)} problems with k={k}")
    _ACTIVE_MODEL = model
    reward_fn = forward_reward if task == "forward" else inverse_reward
    grouped = {}
    prompts = [p["prompt"] for p in problems]
    completions = generate(prompts, n=k)
    for problem, samples in zip(problems, completions):
        rewards = [reward_fn(sample, problem) for sample in samples]
        level = int(problem.get("level", 0))
        split = problem.get("eval_split") or ("seen" if problem.get("skills_seen") else "held_out")
        key = (level, split)
        grouped.setdefault(key, {"p1": [], "pk": []})
        grouped[key]["p1"].append(float(rewards[0] == 1.0) if rewards else 0.0)
        grouped[key]["pk"].append(float(any(r == 1.0 for r in rewards)))

    rows = []
    for (level, split), vals in sorted(grouped.items()):
        rows.append({
            "level": level,
            "split": split,
            "pass@1": sum(vals["p1"]) / len(vals["p1"]),
            f"pass@{k}": sum(vals["pk"]) / len(vals["pk"]),
            "n": len(vals["p1"]),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

def forward_accuracy(model, eval_jsonl=None, k=1, out_name=None):
    """Forward pass@1 by skill and tier on level-1 forward eval examples."""
    eval_jsonl = eval_jsonl or (DATA_DIR / "fwd_l1to4_eval.jsonl")
    problems = [p for p in _read_jsonl(eval_jsonl) if int(p.get("level", 0)) == 1]
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"forward_accuracy_{model_name}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing forward accuracy from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] computing forward accuracy for {len(problems)} level-1 problems")
    global _ACTIVE_MODEL
    _ACTIVE_MODEL = model
    completions = generate([p["prompt"] for p in problems], n=k)
    rows = []
    for problem, samples in zip(problems, completions):
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        rewards = [forward_reward(sample, problem) for sample in samples]
        rows.append({
            "skill": skill,
            "tier": tier,
            "correct": float(any(r == 1.0 for r in rewards)),
        })
    df = pd.DataFrame(rows).groupby(["skill", "tier"], as_index=False).agg(accuracy=("correct", "mean"), n=("correct", "size"))
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

print("[skip] eval utilities are defined; call load_model(), eval_task(), and forward_accuracy() as needed")


In [ ]:
# Cell 4 — Stage 1 forward RFT (stub for EXECUTION_GUIDE Step 4)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 4 stub: Stage 1 forward RFT (stub for EXECUTION_GUIDE Step 4)")


In [ ]:
# Cell 5 — Stage 1.5 inverse baseline (stub for EXECUTION_GUIDE Step 4)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 5 stub: Stage 1.5 inverse baseline (stub for EXECUTION_GUIDE Step 4)")


In [ ]:
# Cell 6 — Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 6 stub: Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)")


In [ ]:
# Cell 7 — Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 7 stub: Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)")


In [ ]:
# Cell 8 — Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 8 stub: Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)")
